In [1]:
import requests
import json
import time
from datetime import datetime, timedelta

class BuscadorContratos:
    """
    Classe para buscar contratos na API de Dados Abertos do governo federal
    """
    def __init__(self):
        self.base_url = "https://dadosabertos.compras.gov.br/modulo-contratos/1_consultarContratos"
        self.headers = {'accept': '*/*','User-Agent': 'Python Script - Busca Contratos'}
    
    def buscar_contratos(self, orgao_codigo, unidade_gestora, 
                        data_inicio=None, data_fim=None,
                        tamanho_pagina=500):
        """
        Busca todos os contratos de um órgão e unidade gestora específicos
        
        Args:
            orgao_codigo: Código do órgão (ex: 12000)
            unidade_gestora: Código da unidade gestora (ex: 090006)
            data_inicio: Data inicial no formato 'YYYY-MM-DD' (padrão: 2015-01-01)
            data_fim: Data final no formato 'YYYY-MM-DD' (padrão: hoje)
            tamanho_pagina: Quantidade de registros por página (padrão: 500)
        
        Returns:
            Lista com todos os contratos encontrados
        """
        # Define datas padrão se não fornecidas
        if data_inicio is None:
            data_inicio = "2015-01-01"
        if data_fim is None:
            data_fim = datetime.now().strftime("%Y-%m-%d")
        todos_contratos = []
        pagina = 1
        total_encontrado = 0
        print(f"Iniciando busca de contratos...")
        print(f"Órgão: {orgao_codigo} | Unidade Gestora: {unidade_gestora}")
        print(f"Período: {data_inicio} até {data_fim}")
        print("-" * 70)
        while True:
            try:
                # Parâmetros da requisição
                params = {'pagina': pagina,'tamanhoPagina': tamanho_pagina,'codigoOrgao': orgao_codigo,'codigoUnidadeGestora': unidade_gestora,
                          'dataVigenciaInicialMin': data_inicio,'dataVigenciaInicialMax': data_fim}
                print(f"\nBuscando página {pagina}...", end=" ")
                
                # Faz a requisição
                response = requests.get(self.base_url,params=params,headers=self.headers,timeout=60)
                
                # Verifica se a requisição foi bem-sucedida
                if response.status_code == 200:
                    dados = response.json()
                    # Extrai os contratos da resposta
                    contratos = dados.get('resultado', [])
                    if not contratos:
                        if pagina == 1:
                            print("Nenhum contrato encontrado.")
                        else:
                            print("Fim dos registros.")
                        break
                    quantidade = len(contratos)
                    todos_contratos.extend(contratos)
                    total_encontrado += quantidade
                    print(f"✓ {quantidade} contratos")
                    print(f"   Total acumulado: {total_encontrado} contratos")
                    # Se retornou menos que o tamanho da página, é a última página
                    if quantidade < tamanho_pagina:
                        print("\nÚltima página alcançada.")
                        break
                    pagina += 1
                    # Delay para não sobrecarregar a API
                    time.sleep(1)
                elif response.status_code == 404:
                    print("✗ Nenhum contrato encontrado (404).")
                    break
                elif response.status_code == 429:
                    print("⚠ Limite de requisições atingido. Aguardando 60 segundos...")
                    time.sleep(60)
                    continue
                else:
                    print(f"✗ Erro {response.status_code}")
                    print(f"Resposta: {response.text[:200]}")
                    break
            except requests.exceptions.Timeout:
                print("✗ Timeout na requisição.")
                print("   A API pode estar lenta. Tentando novamente...")
                time.sleep(5)
                continue
            except requests.exceptions.RequestException as e:
                print(f"✗ Erro na requisição: {e}")
                break
            except json.JSONDecodeError as e:
                print(f"✗ Erro ao decodificar JSON: {e}")
                print(f"Resposta: {response.text[:200]}")
                break
        return todos_contratos
    
    def buscar_por_ano(self, orgao_codigo, unidade_gestora, ano):
        """
        Busca contratos de um ano específico
        """
        data_inicio = f"{ano}-01-01"
        data_fim = f"{ano}-12-31"
        print(f"\n{'='*70}")
        print(f"Buscando contratos do ano {ano}")
        print(f"{'='*70}")  
        contratos = self.buscar_contratos(orgao_codigo, unidade_gestora,data_inicio,data_fim)   
        return contratos
    
    def buscar_multiplos_anos(self, orgao_codigo, unidade_gestora, 
                              ano_inicio=2015, ano_fim=None):
        """
        Busca contratos de múltiplos anos
        """
        if ano_fim is None:
            ano_fim = datetime.now().year
        todos_contratos = []
        for ano in range(ano_inicio, ano_fim + 1):
            contratos = self.buscar_por_ano(orgao_codigo, unidade_gestora, ano)
            if contratos:
                todos_contratos.extend(contratos)
                print(f"\n   Subtotal ano {ano}: {len(contratos)} contratos")
            time.sleep(2)  # Pausa entre anos
        return todos_contratos
       
    def salvar_excel(self, contratos, nome_arquivo=None):
        """
        Salva os contratos em um arquivo Excel com formatação
        """
        try:
            import pandas as pd
            from openpyxl import load_workbook
            from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
            from openpyxl.utils import get_column_letter
        except ImportError:
            print("⚠ Bibliotecas pandas e openpyxl não encontradas.")
            print("  Instalando: pip install pandas openpyxl")
            return None
        
        if not contratos:
            print("Nenhum contrato para salvar em Excel.")
            return None
        
        if nome_arquivo is None:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            nome_arquivo = f"contratos_{timestamp}.xlsx"
        
        # Cria DataFrame
        df = pd.DataFrame(contratos)
        
        # Reordena colunas principais primeiro
        colunas_principais = ['numeroContrato', 'nomeRazaoSocialFornecedor', 'valorGlobal','dataVigenciaInicial', 'dataVigenciaFinal', 'objeto',
                              'nomeCategoria', 'nomeTipo', 'processo']
        
        colunas_ordenadas = [col for col in colunas_principais if col in df.columns]
        outras_colunas = [col for col in df.columns if col not in colunas_principais]
        df = df[colunas_ordenadas + outras_colunas]
        
        # Salva no Excel
        with pd.ExcelWriter(nome_arquivo, engine='openpyxl') as writer:
            df.to_excel(writer, sheet_name='Contratos', index=False)
            
            # Acessa o workbook para formatação
            workbook = writer.book
            worksheet = writer.sheets['Contratos']            
                     
            # Formata valores monetários
            if 'valorGlobal' in df.columns:
                col_idx = df.columns.get_loc('valorGlobal') + 1
                for row in range(2, len(df) + 2):
                    cell = worksheet.cell(row=row, column=col_idx)
                    cell.number_format = 'R$ #,##0.00'
            
            # Formata datas
            for col_name in ['dataVigenciaInicial', 'dataVigenciaFinal', 'dataHoraInclusao']:
                if col_name in df.columns:
                    col_idx = df.columns.get_loc(col_name) + 1
                    for row in range(2, len(df) + 2):
                        cell = worksheet.cell(row=row, column=col_idx)
                        cell.number_format = 'DD/MM/YYYY'
            
            # Congela primeira linha
            worksheet.freeze_panes = 'A2'
        
        return nome_arquivo

def main():
    """
    Função principal
    """
    # Configurações
    ORGAO_CODIGO = "12000"
    UNIDADE_GESTORA = "090006"
    
    # Opções de busca
    
    # BUSCAR TODO O HISTÓRICO (2015 até hoje)
    ANO_INICIO = 2015
    ANO_FIM = datetime.now().year
    
    # Opção alternativa: Buscar apenas últimos anos (descomente para usar)
    # ANO_INICIO = 2023
    # ANO_FIM = datetime.now().year
    
    # Cria o buscador
    buscador = BuscadorContratos()
    
    print(f"{'='*70}")
    print(f"BUSCA DE CONTRATOS - DADOS ABERTOS")
    print(f"{'='*70}")
    print(f"Órgão: {ORGAO_CODIGO}")
    print(f"Unidade Gestora: {UNIDADE_GESTORA}")
    print(f"Período: {ANO_INICIO} até {ANO_FIM}")
    print(f"{'='*70}\n")
    
    # Busca os contratos
    contratos = buscador.buscar_multiplos_anos(
        ORGAO_CODIGO, 
        UNIDADE_GESTORA, 
        ANO_INICIO,
        ANO_FIM
    )
        
    # Salva em arquivos
    if contratos:
        print(f"\n{'='*70}")
        print("SALVANDO ARQUIVOS")
        print(f"{'='*70}")
        buscador.salvar_excel(contratos)
        print(f"\n✓ Processo concluído com sucesso!")
        print(f"\n💡 Dica: Abra o arquivo Excel para visualizar os dados formatados")
    else:
        print("\n⚠ Nenhum contrato foi encontrado para salvar.")


if __name__ == "__main__":
    main()

In [2]:
def main():
    """
    Função principal
    """
    # Configurações
    ORGAO_CODIGO = "12000"
    UNIDADE_GESTORA = "090006"
    
    # Opções de busca
    
    # BUSCAR TODO O HISTÓRICO (2015 até hoje)
    ANO_INICIO = 2015
    ANO_FIM = datetime.now().year
    
    # Opção alternativa: Buscar apenas últimos anos (descomente para usar)
    # ANO_INICIO = 2023
    # ANO_FIM = datetime.now().year
    
    # Cria o buscador
    buscador = BuscadorContratos()
    
    print(f"{'='*70}")
    print(f"BUSCA DE CONTRATOS - DADOS ABERTOS")
    print(f"{'='*70}")
    print(f"Órgão: {ORGAO_CODIGO}")
    print(f"Unidade Gestora: {UNIDADE_GESTORA}")
    print(f"Período: {ANO_INICIO} até {ANO_FIM}")
    print(f"{'='*70}\n")
    
    # Busca os contratos
    contratos = buscador.buscar_multiplos_anos(
        ORGAO_CODIGO, 
        UNIDADE_GESTORA, 
        ANO_INICIO,
        ANO_FIM
    )
        
    # Salva em arquivos
    if contratos:
        print(f"\n{'='*70}")
        print("SALVANDO ARQUIVOS")
        print(f"{'='*70}")
        buscador.salvar_excel(contratos)
        print(f"\n✓ Processo concluído com sucesso!")
        print(f"\n💡 Dica: Abra o arquivo Excel para visualizar os dados formatados")
    else:
        print("\n⚠ Nenhum contrato foi encontrado para salvar.")


if __name__ == "__main__":
    main()

BUSCA DE CONTRATOS - DADOS ABERTOS
Órgão: 12000
Unidade Gestora: 090006
Período: 2015 até 2026


Buscando contratos do ano 2015
Iniciando busca de contratos...
Órgão: 12000 | Unidade Gestora: 090006
Período: 2015-01-01 até 2015-12-31
----------------------------------------------------------------------

Buscando página 1... Nenhum contrato encontrado.

Buscando contratos do ano 2016
Iniciando busca de contratos...
Órgão: 12000 | Unidade Gestora: 090006
Período: 2016-01-01 até 2016-12-31
----------------------------------------------------------------------

Buscando página 1... ✓ 1 contratos
   Total acumulado: 1 contratos

Última página alcançada.

   Subtotal ano 2016: 1 contratos

Buscando contratos do ano 2017
Iniciando busca de contratos...
Órgão: 12000 | Unidade Gestora: 090006
Período: 2017-01-01 até 2017-12-31
----------------------------------------------------------------------

Buscando página 1... Nenhum contrato encontrado.

Buscando contratos do ano 2018
Iniciando busca 

In [ ]:
import requests
import json
import time
from datetime import datetime, timedelta

class BuscadorContratos:
    """
    Classe para buscar contratos na API de Dados Abertos do governo federal
    """
    
    def __init__(self):
        self.base_url = "https://dadosabertos.compras.gov.br/modulo-contratos/1_consultarContratos"
        self.headers = {
            'accept': '*/*',
            'User-Agent': 'Python Script - Busca Contratos'
        }
    
    def buscar_contratos(self, orgao_codigo, unidade_gestora, 
                        data_inicio=None, data_fim=None,
                        tamanho_pagina=500):
        """
        Busca todos os contratos de um órgão e unidade gestora específicos
        
        Args:
            orgao_codigo: Código do órgão (ex: 12000)
            unidade_gestora: Código da unidade gestora (ex: 090006)
            data_inicio: Data inicial no formato 'YYYY-MM-DD' (padrão: 2015-01-01)
            data_fim: Data final no formato 'YYYY-MM-DD' (padrão: hoje)
            tamanho_pagina: Quantidade de registros por página (padrão: 500)
        
        Returns:
            Lista com todos os contratos encontrados
        """
        # Define datas padrão se não fornecidas
        if data_inicio is None:
            data_inicio = "2015-01-01"
        
        if data_fim is None:
            data_fim = datetime.now().strftime("%Y-%m-%d")
        
        todos_contratos = []
        pagina = 1
        total_encontrado = 0
        
        print(f"Iniciando busca de contratos...")
        print(f"Órgão: {orgao_codigo} | Unidade Gestora: {unidade_gestora}")
        print(f"Período: {data_inicio} até {data_fim}")
        print("-" * 70)
        
        while True:
            try:
                # Parâmetros da requisição
                params = {
                    'pagina': pagina,
                    'tamanhoPagina': tamanho_pagina,
                    'codigoOrgao': orgao_codigo,
                    'codigoUnidadeGestora': unidade_gestora,
                    'dataVigenciaInicialMin': data_inicio,
                    'dataVigenciaInicialMax': data_fim
                }
                
                print(f"\nBuscando página {pagina}...", end=" ")
                
                # Faz a requisição
                response = requests.get(
                    self.base_url,
                    params=params,
                    headers=self.headers,
                    timeout=60  # Aumentado para 60 segundos
                )
                
                # Verifica se a requisição foi bem-sucedida
                if response.status_code == 200:
                    dados = response.json()
                    
                    # Extrai os contratos da resposta
                    contratos = dados.get('resultado', [])
                    
                    if not contratos:
                        if pagina == 1:
                            print("Nenhum contrato encontrado.")
                        else:
                            print("Fim dos registros.")
                        break
                    
                    quantidade = len(contratos)
                    todos_contratos.extend(contratos)
                    total_encontrado += quantidade
                    
                    print(f"✓ {quantidade} contratos")
                    print(f"   Total acumulado: {total_encontrado} contratos")
                    
                    # Se retornou menos que o tamanho da página, é a última página
                    if quantidade < tamanho_pagina:
                        print("\nÚltima página alcançada.")
                        break
                    
                    pagina += 1
                    
                    # Delay para não sobrecarregar a API
                    time.sleep(1)
                    
                elif response.status_code == 404:
                    print("✗ Nenhum contrato encontrado (404).")
                    break
                    
                elif response.status_code == 429:
                    print("⚠ Limite de requisições atingido. Aguardando 60 segundos...")
                    time.sleep(60)
                    continue
                    
                else:
                    print(f"✗ Erro {response.status_code}")
                    print(f"Resposta: {response.text[:200]}")
                    break
                    
            except requests.exceptions.Timeout:
                print("✗ Timeout na requisição.")
                print("   A API pode estar lenta. Tentando novamente...")
                time.sleep(5)
                continue
                
            except requests.exceptions.RequestException as e:
                print(f"✗ Erro na requisição: {e}")
                break
            
            except json.JSONDecodeError as e:
                print(f"✗ Erro ao decodificar JSON: {e}")
                print(f"Resposta: {response.text[:200]}")
                break
        
        return todos_contratos
    
    def buscar_por_ano(self, orgao_codigo, unidade_gestora, ano):
        """
        Busca contratos de um ano específico
        """
        data_inicio = f"{ano}-01-01"
        data_fim = f"{ano}-12-31"
        
        print(f"\n{'='*70}")
        print(f"Buscando contratos do ano {ano}")
        print(f"{'='*70}")
        
        contratos = self.buscar_contratos(
            orgao_codigo, 
            unidade_gestora,
            data_inicio,
            data_fim
        )
        
        return contratos
    
    def buscar_contratos_vigentes_apos(self, orgao_codigo, unidade_gestora, 
                                       data_vigencia_final_min, tamanho_pagina=500):
        """
        Busca contratos que têm vigência final após uma data específica
        Útil para capturar contratos antigos que ainda estão vigentes
        """
        todos_contratos = []
        pagina = 1
        total_encontrado = 0
        
        print(f"\nBuscando contratos com vigência após {data_vigencia_final_min}...")
        print("-" * 70)
        
        while True:
            try:
                params = {
                    'pagina': pagina,
                    'tamanhoPagina': tamanho_pagina,
                    'codigoOrgao': orgao_codigo,
                    'codigoUnidadeGestora': unidade_gestora,
                    'dataVigenciaFinalMin': data_vigencia_final_min
                }
                
                print(f"\nBuscando página {pagina}...", end=" ")
                
                response = requests.get(
                    self.base_url,
                    params=params,
                    headers=self.headers,
                    timeout=60
                )
                
                if response.status_code == 200:
                    dados = response.json()
                    contratos = dados.get('resultado', [])
                    
                    if not contratos:
                        if pagina == 1:
                            print("Nenhum contrato encontrado.")
                        else:
                            print("Fim dos registros.")
                        break
                    
                    quantidade = len(contratos)
                    todos_contratos.extend(contratos)
                    total_encontrado += quantidade
                    
                    print(f"✓ {quantidade} contratos")
                    print(f"   Total acumulado: {total_encontrado} contratos")
                    
                    if quantidade < tamanho_pagina:
                        print("\nÚltima página alcançada.")
                        break
                    
                    pagina += 1
                    time.sleep(1)
                    
                elif response.status_code == 404:
                    print("✗ Nenhum contrato encontrado (404).")
                    break
                elif response.status_code == 429:
                    print("⚠ Limite de requisições atingido. Aguardando 60 segundos...")
                    time.sleep(60)
                    continue
                else:
                    print(f"✗ Erro {response.status_code}")
                    break
                    
            except requests.exceptions.Timeout:
                print("✗ Timeout na requisição.")
                print("   A API pode estar lenta. Tentando novamente...")
                time.sleep(5)
                continue
            except requests.exceptions.RequestException as e:
                print(f"✗ Erro na requisição: {e}")
                break
            except json.JSONDecodeError as e:
                print(f"✗ Erro ao decodificar JSON: {e}")
                break
        
        return todos_contratos
    
    def buscar_multiplos_anos(self, orgao_codigo, unidade_gestora, 
                              ano_inicio=2015, ano_fim=None):
        """
        Busca contratos de múltiplos anos
        """
        if ano_fim is None:
            ano_fim = datetime.now().year
        
        todos_contratos = []
        contratos_ids = set()  # Para evitar duplicatas
        
        # Busca por ano de vigência inicial
        for ano in range(ano_inicio, ano_fim + 1):
            contratos = self.buscar_por_ano(orgao_codigo, unidade_gestora, ano)
            if contratos:
                for contrato in contratos:
                    contrato_id = contrato.get('numeroContrato', '') + str(contrato.get('dataVigenciaInicial', ''))
                    if contrato_id not in contratos_ids:
                        todos_contratos.append(contrato)
                        contratos_ids.add(contrato_id)
                print(f"\n   Subtotal ano {ano}: {len(contratos)} contratos")
            time.sleep(2)
        
        # Busca adicional: contratos com vigência final após o último ano
        print(f"\n{'='*70}")
        print(f"Busca complementar: contratos vigentes após {ano_fim}")
        print(f"{'='*70}")
        
        contratos_vigentes = self.buscar_contratos_vigentes_apos(
            orgao_codigo,
            unidade_gestora,
            f"{ano_fim}-01-01"
        )
        
        if contratos_vigentes:
            novos_contratos = 0
            for contrato in contratos_vigentes:
                contrato_id = contrato.get('numeroContrato', '') + str(contrato.get('dataVigenciaInicial', ''))
                if contrato_id not in contratos_ids:
                    todos_contratos.append(contrato)
                    contratos_ids.add(contrato_id)
                    novos_contratos += 1
            
            print(f"\n   Contratos adicionais encontrados: {novos_contratos}")
            print(f"   (Contratos antigos ainda vigentes em {ano_fim})")
        
        return todos_contratos
    
    def salvar_json(self, contratos, nome_arquivo=None):
        """
        Salva os contratos em um arquivo JSON
        """
        if nome_arquivo is None:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            nome_arquivo = f"contratos_{timestamp}.json"
        
        with open(nome_arquivo, 'w', encoding='utf-8') as f:
            json.dump(contratos, f, ensure_ascii=False, indent=2)
        
        print(f"\n✓ Dados salvos em: {nome_arquivo}")
        tamanho_mb = len(json.dumps(contratos)) / (1024 * 1024)
        print(f"  Tamanho: {tamanho_mb:.2f} MB")
        
        return nome_arquivo
    
    def salvar_csv(self, contratos, nome_arquivo=None):
        """
        Salva os contratos em um arquivo CSV
        """
        import csv
        
        if not contratos:
            print("Nenhum contrato para salvar em CSV.")
            return None
        
        if nome_arquivo is None:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            nome_arquivo = f"contratos_{timestamp}.csv"
        
        # Obtém todas as chaves possíveis
        todas_chaves = set()
        for contrato in contratos:
            todas_chaves.update(contrato.keys())
        
        todas_chaves = sorted(todas_chaves)
        
        with open(nome_arquivo, 'w', encoding='utf-8-sig', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=todas_chaves, delimiter=';')
            writer.writeheader()
            writer.writerows(contratos)
        
        print(f"✓ CSV salvo em: {nome_arquivo}")
        return nome_arquivo
    
    def salvar_excel(self, contratos, nome_arquivo=None):
        """
        Salva os contratos em um arquivo Excel com formatação
        """
        try:
            import pandas as pd
            from openpyxl import load_workbook
            from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
            from openpyxl.utils import get_column_letter
        except ImportError:
            print("⚠ Bibliotecas pandas e openpyxl não encontradas.")
            print("  Instalando: pip install pandas openpyxl")
            return None
        
        if not contratos:
            print("Nenhum contrato para salvar em Excel.")
            return None
        
        if nome_arquivo is None:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            nome_arquivo = f"contratos_{timestamp}.xlsx"
        
        # Cria DataFrame MANTENDO A ORDEM ORIGINAL DOS CAMPOS
        df = pd.DataFrame(contratos)
        
        # Salva no Excel
        with pd.ExcelWriter(nome_arquivo, engine='openpyxl') as writer:
            df.to_excel(writer, sheet_name='Contratos', index=False)
            
            # Acessa o workbook para formatação
            workbook = writer.book
            worksheet = writer.sheets['Contratos']
            
            # Estilo do cabeçalho
            header_fill = PatternFill(start_color='366092', end_color='366092', fill_type='solid')
            header_font = Font(bold=True, color='FFFFFF', size=11)
            border = Border(
                left=Side(style='thin'),
                right=Side(style='thin'),
                top=Side(style='thin'),
                bottom=Side(style='thin')
            )
            
            # Aplica formatação no cabeçalho
            for cell in worksheet[1]:
                cell.fill = header_fill
                cell.font = header_font
                cell.alignment = Alignment(horizontal='center', vertical='center')
                cell.border = border
            
            # Ajusta largura das colunas
            for column in worksheet.columns:
                max_length = 0
                column_letter = get_column_letter(column[0].column)
                
                for cell in column:
                    try:
                        if len(str(cell.value)) > max_length:
                            max_length = len(str(cell.value))
                    except:
                        pass
                
                adjusted_width = min(max_length + 2, 80)
                worksheet.column_dimensions[column_letter].width = adjusted_width
            
            # Formata valores monetários
            if 'valorGlobal' in df.columns:
                col_idx = df.columns.get_loc('valorGlobal') + 1
                for row in range(2, len(df) + 2):
                    cell = worksheet.cell(row=row, column=col_idx)
                    cell.number_format = 'R$ #,##0.00'
            
            # Formata datas
            for col_name in ['dataVigenciaInicial', 'dataVigenciaFinal', 'dataHoraInclusao']:
                if col_name in df.columns:
                    col_idx = df.columns.get_loc(col_name) + 1
                    for row in range(2, len(df) + 2):
                        cell = worksheet.cell(row=row, column=col_idx)
                        cell.number_format = 'DD/MM/YYYY'
            
            # Congela primeira linha
            worksheet.freeze_panes = 'A2'
        
        print(f"✓ Excel salvo em: {nome_arquivo}")
        tamanho_mb = len(df.to_json()) / (1024 * 1024)
        print(f"  Tamanho: {tamanho_mb:.2f} MB")
        print(f"  Registros: {len(df)} contratos")
        
        return nome_arquivo
    
    def exibir_resumo(self, contratos):
        """
        Exibe um resumo dos contratos encontrados
        """
        print("\n" + "=" * 70)
        print("RESUMO DOS CONTRATOS ENCONTRADOS")
        print("=" * 70)
        print(f"Total de contratos: {len(contratos)}")
        
        if contratos:
            # Estatísticas básicas
            valores = [c.get('valorGlobal', 0) for c in contratos if c.get('valorGlobal')]
            if valores:
                print(f"\nValores:")
                print(f"  - Valor total: R$ {sum(valores):,.2f}".replace(',', '_').replace('.', ',').replace('_', '.'))
                print(f"  - Valor médio: R$ {sum(valores)/len(valores):,.2f}".replace(',', '_').replace('.', ',').replace('_', '.'))
                print(f"  - Valor máximo: R$ {max(valores):,.2f}".replace(',', '_').replace('.', ',').replace('_', '.'))
                print(f"  - Valor mínimo: R$ {min(valores):,.2f}".replace(',', '_').replace('.', ',').replace('_', '.'))
            
            # Contratos por categoria
            categorias = {}
            for c in contratos:
                cat = c.get('nomeCategoria', 'Sem categoria')
                categorias[cat] = categorias.get(cat, 0) + 1
            
            if categorias:
                print(f"\nContratos por categoria:")
                for cat, qtd in sorted(categorias.items(), key=lambda x: x[1], reverse=True):
                    print(f"  - {cat}: {qtd}")
            
            # Exemplo do primeiro contrato
            print(f"\n{'='*70}")
            print("EXEMPLO DO PRIMEIRO CONTRATO:")
            print(f"{'='*70}")
            primeiro = contratos[0]
            print(f"Número: {primeiro.get('numeroContrato')}")
            print(f"Fornecedor: {primeiro.get('nomeRazaoSocialFornecedor')}")
            print(f"Valor: R$ {primeiro.get('valorGlobal', 0):,.2f}".replace(',', '_').replace('.', ',').replace('_', '.'))
            print(f"Vigência: {primeiro.get('dataVigenciaInicial')} até {primeiro.get('dataVigenciaFinal')}")
            print(f"Objeto: {primeiro.get('objeto', '')[:150]}...")
            
            print(f"\n{'='*70}")
            print(f"Campos disponíveis ({len(primeiro.keys())} campos):")
            print(f"{'='*70}")
            for i, campo in enumerate(sorted(primeiro.keys()), 1):
                print(f"{i:2d}. {campo}")


def main():
    """
    Função principal
    """
    # Configurações
    ORGAO_CODIGO = "12000"
    UNIDADE_GESTORA = "090006"
    
    # Opções de busca
    
    # BUSCAR TODO O HISTÓRICO (2015 até hoje)
    ANO_INICIO = 2015
    ANO_FIM = datetime.now().year
    
    # Opção alternativa: Buscar apenas últimos anos (descomente para usar)
    # ANO_INICIO = 2023
    # ANO_FIM = datetime.now().year
    
    # Cria o buscador
    buscador = BuscadorContratos()
    
    print(f"{'='*70}")
    print(f"BUSCA DE CONTRATOS - DADOS ABERTOS")
    print(f"{'='*70}")
    print(f"Órgão: {ORGAO_CODIGO}")
    print(f"Unidade Gestora: {UNIDADE_GESTORA}")
    print(f"Período: {ANO_INICIO} até {ANO_FIM}")
    print(f"{'='*70}\n")
    
    # Busca os contratos
    contratos = buscador.buscar_multiplos_anos(
        ORGAO_CODIGO, 
        UNIDADE_GESTORA, 
        ANO_INICIO,
        ANO_FIM
    )
    
    # Exibe resumo
    buscador.exibir_resumo(contratos)
    
    # Salva em arquivos
    if contratos:
        print(f"\n{'='*70}")
        print("SALVANDO ARQUIVOS")
        print(f"{'='*70}")
        buscador.salvar_json(contratos)
        buscador.salvar_excel(contratos)
        print(f"\n✓ Processo concluído com sucesso!")
        print(f"\n💡 Dica: Abra o arquivo Excel para visualizar os dados formatados")
    else:
        print("\n⚠ Nenhum contrato foi encontrado para salvar.")


if __name__ == "__main__":
    main()